In [2]:
import pickle
import numpy as np
import networkx as nx
from scipy.stats import ks_2samp

In [3]:
def compute_2d_ks(test_points, gen_points):
    ks_x = ks_2samp(test_points[:, 0], gen_points[:, 0]).statistic
    ks_y = ks_2samp(test_points[:, 1], gen_points[:, 1]).statistic
    return (ks_x + ks_y) / 2

In [4]:
# -------- Node Degree Behavior Metric (2D KS using Q1 and Q3 of degree) -------- #
def extract_pagerank_q1_q3_points(graph_seq):
    q1_q3_points = []
    for g in graph_seq:
        pagerank_scores = np.array([d for _, d in nx.pagerank(g).items()])
        if len(pagerank_scores) < 4:
            continue
        q1 = np.percentile(pagerank_scores, 25)
        q3 = np.percentile(pagerank_scores, 75)
        q1_q3_points.append([q1, q3])
    return np.array(q1_q3_points)

In [5]:
def run_pagerank_eval(test_graph_seqs, gen_graph_seqs):
    n = min(len(test_graph_seqs), len(gen_graph_seqs))
    ks_scores = []
    for i in range(n):
        # print(f'test_graphs[{i}]: {test_graphs[i]}')
        test_points = extract_pagerank_q1_q3_points(test_graph_seqs[i])
        gen_points = extract_pagerank_q1_q3_points(gen_graph_seqs[i])
        if len(test_points) == 0 or len(gen_points) == 0:
            continue
        ks = compute_2d_ks(test_points, gen_points)
        ks_scores.append(ks)
    return np.mean(ks_scores) if ks_scores else None

In [6]:
def pagerank_KS_evaluator(generated_graph_seqs, reference_graph_seqs):
    print(f'Computing pagerank KS...')
    pagerank_ks = run_pagerank_eval(generated_graph_seqs, reference_graph_seqs)
    return pagerank_ks

In [ ]:
dataset_list = ['digg','superuser','twitter','wiki-vote']
model_list = ['TagGen','AGE','DAMNET','DYMOND']
for dataset_name in dataset_list:
    print(f'dataset: {dataset_name}')
    for model_name in model_list:
        print(f'model: {model_name}')
        print('Loading testing and sampled graph data...')
        test_graph_path = f'../test_and_generated_graphs/{dataset_name}/{model_name}/test_graphs.pkl' # fill in the corresponding info
        sampled_graph_path = f'../test_and_generated_graphs/{dataset_name}/{model_name}/sampled_ts.pkl' # fill in the corresponding info
        with open(sampled_graph_path,'rb') as f_gen:
            sampled_graphs = pickle.load(f_gen)
            print(f'There are {len(sampled_graphs)} generated temporal graph sequences of length {len(sampled_graphs[0])}.')
        with open(test_graph_path,'rb') as f_test:
            test_graphs = pickle.load(f_test)
            print(f'There are {len(test_graphs)} testing temporal graph sequences of length {len(test_graphs[0])}.')
        
        num_seqs = min(len(test_graphs), len(sampled_graphs))
        pagerank_ks = pagerank_KS_evaluator(sampled_graphs, test_graphs)
        print(f'Pagerank KS: {pagerank_ks}')

dataset: digg
model: TagGen
Loading testing and sampled graph data...
There are 5 generated temporal graph sequences of length 4.
There are 25000 testing temporal graph sequences of length 4.
Computing pagerank KS...
Pagerank KS: 0.925
model: AGE
Loading testing and sampled graph data...
There are 25000 generated temporal graph sequences of length 4.
There are 25000 testing temporal graph sequences of length 4.
Computing pagerank KS...
Pagerank KS: 0.89673
model: DAMNET
Loading testing and sampled graph data...
There are 25000 generated temporal graph sequences of length 4.
There are 25000 testing temporal graph sequences of length 4.
Computing pagerank KS...
Pagerank KS: 0.845255
model: DYMOND
Loading testing and sampled graph data...
There are 25000 generated temporal graph sequences of length 4.
There are 25000 testing temporal graph sequences of length 4.
Computing pagerank KS...
Pagerank KS: 0.7583033333333334
dataset: superuser
model: TagGen
Loading testing and sampled graph data

In [9]:
dataset_list = ['superuser']
model_name = 'MulDyDiff'
for dataset_name in dataset_list:
    print(f'dataset: {dataset_name}')
    print('Loading testing and sampled graph data...')
    test_graph_path = f'../test_and_generated_graphs/{dataset_name}/{model_name}/test_graphs.pkl' # fill in the corresponding info
    sampled_graph_path = f'../test_and_generated_graphs/{dataset_name}/{model_name}/sampled_ts.pkl' # fill in the corresponding info
    with open(sampled_graph_path,'rb') as f_gen:
        sampled_graphs = pickle.load(f_gen)
        print(f'There are {len(sampled_graphs)} generated temporal graph sequences of length {len(sampled_graphs[0])}.')
    with open(test_graph_path,'rb') as f_test:
        test_graphs = pickle.load(f_test)
        print(f'There are {len(test_graphs)} testing temporal graph sequences of length {len(test_graphs[0])}.')
    num_seqs = min(len(test_graphs), len(sampled_graphs))
    pagerank_ks = pagerank_KS_evaluator(sampled_graphs, test_graphs)
    print(f'Pagerank KS: {pagerank_ks}')
        
    

dataset: superuser
Loading testing and sampled graph data...
There are 1600 generated temporal graph sequences of length 4.
There are 16000 testing temporal graph sequences of length 4.
Computing pagerank KS...
Pagerank KS: 0.839609375
